# 📈 Quant 101: Interactive Research & Learning Lab

Welcome! This notebook provides an interactive step-by-step tutorial on building quantitative trading models.

### 📑 Table of Contents
1. **Step 1: Data Pipeline & Time Series Math** (Fetching data with `yfinance`, Log Returns $R_t$, Rolling Volatility $\sigma$)
2. **Step 2: Signal Generation** (SMA 50/200 Crossover Momentum Strategy)
3. **Step 3: Statistical Arbitrage & Pairs Trading** (OLS Regression, ADF Stationarity Test, $Z$-Score Spread)
4. **Step 4: Vectorized Backtesting Engine** (`vectorbt` simulation with transaction fees)
5. **Step 5: Risk Analytics & Monte Carlo Simulation** (Sharpe Ratio, Drawdown, 10k Monte Carlo Stress Testing)

In [ ]:
# ☁️ Google Colab Installation Cell
# Run this cell if opening in Google Colab:
!pip install -q --upgrade scipy numpy pandas yfinance statsmodels vectorbt quantstats matplotlib pyarrow


--- 
## 🔹 Step 1: Data Pipeline & Time-Series Mathematics

### Mathematical Formulas:
1. **Logarithmic Returns ($R_t$):**
$$R_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$

2. **Rolling 20-Day Volatility ($\sigma_{20}$):**
$$\sigma_{20} = \sqrt{\frac{1}{19} \sum_{i=1}^{20} (R_i - \bar{R})^2}$$

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)

# 1. Download 10 years of S&P 500 (^GSPC) data
print("Downloading historical data from Yahoo Finance...")
df = yf.download('^GSPC', period='10y', progress=False)
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Clean missing data
df = df[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.ffill(inplace=True)
df.bfill(inplace=True)

# 2. Calculate Log Returns & Rolling Volatility
df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
df['Volatility_20d'] = df['Log_Return'].rolling(window=20).std()

# Display recent rows
df.tail()

In [ ]:
# Plot S&P 500 Price and 20-Day Rolling Volatility
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(df.index, df['Close'], color='#1f77b4', linewidth=1.5, label='S&P 500 Close Price')
ax1.set_title('S&P 500 (^GSPC) Historical Close Price (10 Years)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left')
ax1.grid(True)

ax2.plot(df.index, df['Volatility_20d'], color='#ff7f0e', linewidth=1.2, label='20-Day Rolling Volatility (σ)')
ax2.set_title('20-Day Rolling Sample Volatility', fontsize=12, fontweight='bold')
ax2.set_ylabel('Volatility')
ax2.set_xlabel('Date')
ax2.legend(loc='upper left')
ax2.grid(True)

plt.tight_layout()
plt.show()

--- 
## 🔹 Step 2: Signal Generation (Momentum Strategy)

### Trend Following Rules:
- **50-Day Moving Average ($	ext{SMA}_{50}$):** Short-term momentum trend.
- **200-Day Moving Average ($	ext{SMA}_{200}$):** Long-term baseline trend.
- **Signal Rule:**
  - $\text{SMA}_{50} > \text{SMA}_{200} \implies \text{Signal} = 1$ (Golden Cross / Buy)
  - $\text{SMA}_{50} < \text{SMA}_{200} \implies \text{Signal} = -1$ (Death Cross / Sell)

In [ ]:
# Compute 50-day and 200-day Simple Moving Averages
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['SMA_200'] = df['Close'].rolling(window=200).mean()

# Generate Buy (1) and Sell (-1) signals
df['Signal'] = 0
df.loc[df['SMA_50'] > df['SMA_200'], 'Signal'] = 1
df.loc[df['SMA_50'] < df['SMA_200'], 'Signal'] = -1

# Plot Moving Averages and Buy/Sell Signal regions
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='Close Price', alpha=0.5, color='gray')
plt.plot(df.index, df['SMA_50'], label='50-Day SMA', color='green', linewidth=1.8)
plt.plot(df.index, df['SMA_200'], label='200-Day SMA', color='red', linewidth=1.8)

# Fill signal regions
plt.fill_between(df.index, df['Close'].min(), df['Close'].max(), where=(df['Signal'] == 1), color='green', alpha=0.1, label='Long Position (Buy)')
plt.fill_between(df.index, df['Close'].min(), df['Close'].max(), where=(df['Signal'] == -1), color='red', alpha=0.1, label='Short Position (Sell)')

plt.title('SMA Crossover Strategy Signals (Golden Cross / Death Cross)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

--- 
## 🔹 Step 3: Statistical Modeling (Pairs Trading)

### Mathematical Formulas:
1. **OLS Linear Regression:** $Y_t = \beta X_t + \alpha + \epsilon_t$
2. **Spread Residual:** $\text{Spread}_t = Y_t - (\beta X_t + \alpha)$
3. **Normalized $Z$-Score:** $Z_t = \frac{\text{Spread}_t - \mu_{\text{spread}}}{\sigma_{\text{spread}}}$

In [ ]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

# 1. Download correlated stock pair: Pepsi (PEP) and Coca-Cola (KO)
pair_data = yf.download(['PEP', 'KO'], period='10y', progress=False)['Close'].dropna()
Y = pair_data['PEP']
X = pair_data['KO']

# 2. OLS Regression
X_const = sm.add_constant(X)
model = sm.OLS(Y, X_const).fit()
alpha, beta = model.params['const'], model.params['KO']
spread = Y - (beta * X + alpha)

# 3. ADF Stationarity Test
adf_res = adfuller(spread)
print(f"Hedge Ratio (Beta): {beta:.4f}")
print(f"ADF Statistic:      {adf_res[0]:.4f} (p-value: {adf_res[1]:.4f})")

# 4. Calculate Z-score of spread
z_score = (spread - spread.rolling(30).mean()) / spread.rolling(30).std()

# Plot Spread and Z-Score Thresholds (+2 / -2)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
ax1.plot(spread.index, spread, color='purple', label='Price Spread (PEP - Beta*KO)')
ax1.set_title('Pairs Spread', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True)

ax2.plot(z_score.index, z_score, color='darkblue', label='Normalized Z-Score')
ax2.axhline(2.0, color='red', linestyle='--', label='Short Signal (+2.0)')
ax2.axhline(-2.0, color='green', linestyle='--', label='Long Signal (-2.0)')
ax2.axhline(0, color='black', linestyle=':', alpha=0.7)
ax2.set_title('Spread Z-Score Mean-Reversion Signals', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

--- 
## 🔹 Step 4: Vectorized Backtesting Engine (`vectorbt`)

We test our SMA strategy starting with **$10,000 initial capital** and deducting a **0.1% transaction friction fee** on every trade.

In [ ]:
import vectorbt as vbt

entries = df['Signal'] == 1
exits = df['Signal'] == -1

# Run vectorized backtest simulation
portfolio = vbt.Portfolio.from_signals(
    df['Close'],
    entries=entries,
    exits=exits,
    init_cash=10000.0,
    fees=0.001, # 0.1% fee per trade
    freq='D'
)

# Plot Equity Curve vs Benchmark
portfolio.plot().show()

--- 
## 🔹 Step 5: Risk Analytics & Monte Carlo Simulation

### Mathematical Formulas:
1. **Sharpe Ratio ($S$):** $S = \frac{\mathbb{E}[R_p - R_f]}{\sigma_p}$
2. **Maximum Drawdown ($MDD$):** $MDD = \max_{\tau \le t} \left( \frac{P_{\text{peak}} - P_\tau}{P_{\text{peak}}} \right)$
3. **Monte Carlo Stress Test:** 10,000 random trade order permutations to calculate 95% Value-at-Risk.

In [ ]:
returns = df['Log_Return'].dropna().values
n_simulations = 10000
sample_size = 252 # 1 Trading Year

np.random.seed(42)
final_returns = []
max_drawdowns = []

for _ in range(n_simulations):
    sim_returns = np.random.choice(returns, size=sample_size, replace=True)
    equity_curve = np.cumprod(1 + sim_returns)
    final_returns.append(equity_curve[-1] - 1)
    
    running_max = np.maximum.accumulate(equity_curve)
    drawdowns = (equity_curve - running_max) / running_max
    max_drawdowns.append(drawdowns.min())

final_returns = np.array(final_returns)
max_drawdowns = np.array(max_drawdowns)

# Plot Monte Carlo Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(final_returns * 100, bins=50, color='#1f77b4', edgecolor='black', alpha=0.7)
ax1.axvline(np.percentile(final_returns, 5) * 100, color='red', linestyle='--', label=f'95% VaR ({np.percentile(final_returns, 5)*100:.1f}%)')
ax1.set_title('Monte Carlo: 1-Year Expected Return Distribution (%)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Return (%)')
ax1.set_ylabel('Frequency')
ax1.legend()
ax1.grid(True)

ax2.hist(max_drawdowns * 100, bins=50, color='#d62728', edgecolor='black', alpha=0.7)
ax2.axvline(np.percentile(max_drawdowns, 5) * 100, color='black', linestyle='--', label=f'Worst 5% Drawdown ({np.percentile(max_drawdowns, 5)*100:.1f}%)')
ax2.set_title('Monte Carlo: Max Drawdown Distribution (%)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Max Drawdown (%)')
ax2.set_ylabel('Frequency')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()